# I Built a RAG System That Cut Incident Response from 45 Minutes to 3 — Here's Every Line"

In [1]:
 !pip install langchain-community langchain-openai chromadb tiktoken

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached chromadb-1.5.5-cp39-abi3-win_amd64.whl.metadata (7.3 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_api-1.40.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.40.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.40.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached kubernetes-35.0.0-py2.py

In [5]:
!pip install langchain-community langchain-text-splitters

In [3]:

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("service_x_runbook.txt")
documents = loader.load()


splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     
    chunk_overlap=50     
                         
)

chunks = splitter.split_documents(documents)


print(chunks[0].page_content)
print(f"Total chunks: {len(chunks)}")

SERVICE X — INCIDENT RUNBOOK
Last Updated: 2024-11-01
Owner: Platform Infrastructure Team

SECTION 1: SERVICE OVERVIEW
Total chunks: 12


In [1]:
# First, install the required packages
!pip install langchain-community langchain-text-splitters

# Then import the modules
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("service_x_runbook.txt")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     
    chunk_overlap=50     
)

chunks = splitter.split_documents(documents)

print(chunks[0].page_content)
print(f"Total chunks: {len(chunks)}")

SERVICE X — INCIDENT RUNBOOK
Last Updated: 2024-11-01
Owner: Platform Infrastructure Team

SECTION 1: SERVICE OVERVIEW
Total chunks: 12


In [2]:
# First, install the required packages
!pip install langchain-community langchain-text-splitters

# Then import the modules
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("service_x_runbook.txt")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     
    chunk_overlap=50     
)

chunks = splitter.split_documents(documents)

print(chunks[0].page_content)
print(f"Total chunks: {len(chunks)}")

SERVICE X — INCIDENT RUNBOOK
Last Updated: 2024-11-01
Owner: Platform Infrastructure Team

SECTION 1: SERVICE OVERVIEW
Total chunks: 12


In [21]:
import os
os.environ["OPENAI_API_KEY"] ="................"

In [22]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)



vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db" 
                                   
)

print("Vector store created. Chunks embedded and saved.")

Vector store created. Chunks embedded and saved.


In [23]:
retriever = vectorstore.as_retriever(
    search_type="similarity",search_kwargs={"k": 4})

In [25]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate    


llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)



prompt_template = ChatPromptTemplate.from_template("""
You are an on-call assistant for infrastructure incidents.
You have been given relevant excerpts from internal runbooks.
Answer the engineer's question using ONLY the provided context.
If the context doesn't contain enough information to answer,
say "I don't have enough runbook context for this — escalate to Tier 2."
Do NOT hallucinate or guess.

Context from runbooks:
{context}

Engineer's question:
{question}

Answer:
""")


In [26]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)
 

rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever | format_docs
    }
    | prompt_template  
    | llm            
    | StrOutputParser()
                      
)

In [27]:
question = "Service X is throwing connection refused errors. What are the first steps?"

response = rag_chain.invoke(question)

print("RAG Response")
print(response)
retrieved_chunks = retriever.invoke(question)
print("\n Chunks Used")
for i, chunk in enumerate(retrieved_chunks):
    print(f"\n[Chunk {i+1}]")
    print(chunk.page_content)

=== RAG Response ===
1. Check if the Service X process is running by executing: `systemctl status service-x`.
2. If the process is inactive, view the last 100 log lines with: `journalctl -u service-x -n 100`.
3. Check for an out-of-memory kill by running: `dmesg | grep -i oom`.
4. If the process has crashed, restart it using: `systemctl restart service-x`.
5. If the restart fails repeatedly, escalate to Tier 2.

=== Chunks Used ===

[Chunk 1]
Service X is the primary API gateway for all internal microservices.
It handles authentication, rate limiting, and request routing.
It runs on port 8080 and communicates with downstream services via gRPC on port 9090.
Service X depends on Redis for session caching and PostgreSQL for persistent auth tokens.

SECTION 2: CONNECTION REFUSED ERRORS

[Chunk 2]
Symptom: Clients receive "connection refused" on port 8080.

Root Cause 1 — Service X process crashed:
  - Check if the process is running: `systemctl status service-x`
  - If inactive: `journalct

In [30]:
#TEST 1 — Should pull from service_x_runbook.txt
question = "Service X is throwing connection refused errors, what do I do?"
response = rag_chain.invoke(question)
print(response)

To address the "connection refused" errors for Service X, follow these steps:

1. **Check if the Service X process is running**: Use the command `systemctl status service-x`.
   - If the process is inactive, check the last 100 log lines with `journalctl -u service-x -n 100` to identify any issues.
   - Look for common causes such as an out-of-memory kill by checking with `dmesg | grep -i oom`.
   - If the process has crashed, restart it using `systemctl restart service-x`. If the restart fails repeatedly, escalate to Tier 2.

2. **Check for Redis connection pool exhaustion**: Look for logs indicating "Redis: max connections reached".
   - Use `redis-cli info clients | grep connected_clients` to check the number of connected clients.
   - If the number of connected clients exceeds 500, flush idle connections with `redis-cli client kill skipme yes ID 0 ADDR 0.0.0.0:0 MAXAGE 300`.
   - If this issue recurs, consider increasing the pool size by setting `REDIS_MAX_CONNECTIONS=1000` in `/etc

In [31]:
#TEST 2 — Should pull from redis_runbook.txt
question = "Redis is not responding and the process seems crashed"
response = rag_chain.invoke(question)
print(response)


I don't have enough runbook context for this — escalate to Tier 2.


In [32]:
question = "Redis is not responding and the process seems crashed"
chunks = retriever.invoke(question)

for i, chunk in enumerate(chunks):
    print(f"\n[Chunk {i+1}]")
    print(f"Source: {chunk.metadata}")
    print(f"Content: {chunk.page_content}")
    print("-"*40)


[Chunk 1]
Source: {'source': '/content/drive/MyDrive/service_x_runbook.txt'}
Content: Root Cause 2 — Redis connection pool exhausted:
  - Symptom: Service X logs show "Redis: max connections reached"
  - Check Redis connections: `redis-cli info clients | grep connected_clients`
  - If connected_clients > 500, flush idle connections: `redis-cli client kill skipme yes ID 0 ADDR 0.0.0.0:0 MAXAGE 300`
  - Increase pool size in config if recurring: set REDIS_MAX_CONNECTIONS=1000 in /etc/service-x/env
----------------------------------------

[Chunk 2]
Source: {'source': '/content/drive/MyDrive/service_x_runbook.txt'}
Content: Symptom: Clients receive "connection refused" on port 8080.

Root Cause 1 — Service X process crashed:
  - Check if the process is running: `systemctl status service-x`
  - If inactive: `journalctl -u service-x -n 100` to view last 100 log lines
  - Common cause: out-of-memory kill. Check: `dmesg | grep -i oom`
  - Fix: `systemctl restart service-x`
  - If restart fai

In [35]:
import shutil
import os
CHROMA_PATH = "/tmp/chroma_db"

# Delete if exists
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)
    print("Old vector store deleted")

# Re-embed with writable path
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_PATH
)
print("Vector store rebuilt")

# Update your retriever to point to the new path too
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)
print("Retriever updated")

✅ Vector store rebuilt
✅ Retriever updated


In [36]:
CHROMA_PATH = "/content/drive/MyDrive/chroma_db"

In [37]:
question = "Redis is not responding and the process seems crashed"
response = rag_chain.invoke(question)
print(response)

I don't have enough runbook context for this — escalate to Tier 2.


In [38]:
print(f"Total chunks in vectorstore: {vectorstore._collection.count()}")
results = vectorstore.get()
sources = set(m['source'] for m in results['metadatas'])
print(f"Sources in vectorstore: {sources}")

# Check 3 — raw retrieval for the Redis question
question = "Redis is not responding and the process seems crashed"
chunks = retriever.invoke(question)
for i, chunk in enumerate(chunks):
    print(f"\n[Chunk {i+1}] Source: {chunk.metadata['source']}")
    print(chunk.page_content[:150])

Total chunks in vectorstore: 34
Sources in vectorstore: {'/content/drive/MyDrive/service_x_runbook.txt', '/content/drive/MyDrive/database_runbook.txt', '/content/drive/MyDrive/redis_runbook.txt'}

[Chunk 1] Source: /content/drive/MyDrive/redis_runbook.txt
Redis runs as a single primary instance on redis.internal (192.168.1.20), port 6379.
It is used for: session caching, rate limiting counters, and dist

[Chunk 2] Source: /content/drive/MyDrive/redis_runbook.txt
Step 2 — Common crash cause — OOM kill:
  - `dmesg | grep -i "redis"`
  - If OOM kill found, Redis exceeded system memory (not just its maxmemory sett

[Chunk 3] Source: /content/drive/MyDrive/redis_runbook.txt
Symptom: Applications report Redis connection timeouts or refused connections.

Step 1 — Check Redis process:
  - `systemctl status redis`
  - If not 

[Chunk 4] Source: /content/drive/MyDrive/redis_runbook.txt
REDIS CACHE — INCIDENT RUNBOOK
Last Updated: 2024-09-20
Owner: Platform Infrastructure Team

S


In [40]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
test = llm.invoke("Say hello")
print("LLM works:", test.content)


question = "Redis is not responding and the process seems crashed"
chunks = retriever.invoke(question)

from langchain_text_splitters import RecursiveCharacterTextSplitter

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

filled_prompt = prompt_template.invoke({
    "context": format_docs(chunks),
    "question": question
})
print("\n FILLED PROMPT ")
print(filled_prompt.to_string())

# Test 3 — send filled prompt directly to LLM, bypassing the chain
direct_response = llm.invoke(filled_prompt)
print("\n DIRECT LLM RESPONSE")
print(direct_response.content)

LLM works: Hello! How can I assist you today?

--- FILLED PROMPT ---
Human: 
You are an on-call assistant for infrastructure incidents.
You have been given relevant excerpts from internal runbooks.
Answer the engineer's question using ONLY the provided context.
If the context doesn't contain enough information to answer,
say "I don't have enough runbook context for this — escalate to Tier 2."
Do NOT hallucinate or guess.

Context from runbooks:
Redis runs as a single primary instance on redis.internal (192.168.1.20), port 6379.
It is used for: session caching, rate limiting counters, and distributed locks.
Max memory: 8GB with eviction policy: allkeys-lru
Persistence: RDB snapshots every 15 minutes + AOF with everysec fsync

SECTION 2: REDIS DOWN / NOT RESPONDING

---

Step 2 — Common crash cause — OOM kill:
  - `dmesg | grep -i "redis"`
  - If OOM kill found, Redis exceeded system memory (not just its maxmemory setting).
  - Check actual memory used: `redis-cli info memory | grep used

The direct LLM call returned the perfect answer, which means your prompt, chunks, and LLM are all correct. The only thing broken was the chain still pointing at the old vectorstore/retriever from before you rebuilt everything. This rebuilds it fresh with the current retriever.

In [41]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever | format_docs
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

# Test it
response = rag_chain.invoke("Redis is not responding and the process seems crashed")
print(response)

Check the Redis process status with `systemctl status redis`. If it is not running, use `journalctl -u redis -n 200` to check why it stopped. If you find that it was OOM killed, you can temporarily fix the issue by restarting Redis with `systemctl start redis`. For a permanent fix, consider reducing the maxmemory or upgrading the instance.


In [42]:
question = "what is whole datasets andf this rag you are helping me with?"
response = rag_chain.invoke(question)
print(response)

I don't have enough runbook context for this — escalate to Tier 2.


In [45]:
question = input("ask me a question")
response = rag_chain.invoke(question)
print(response)

I don't have enough runbook context for this — escalate to Tier 2.
